## <span style="color: #ff7402">Atelier Préparation Données Images</span>

Contexte 
Une entreprise souhaite développer un système d’intelligence artificielle capable de reconnaître 
automatiquement le type de déchet présent sur une photographie afin d'améliorer le tri des 
déchets. 

Le modèle devra classer chaque image dans l'une des catégories suivantes : 

* cardboard : cartons ondulés, cartons plats, … 
* plastic : bouteilles, emballages plastiques... 
* paper : feuilles,  journaux... 
* glass : bouteilles et objets en verre... 
* metal : canettes, boîtes métalliques... 
* trash : emballages bonbons, tasses jetables, ... 

Le problème est que les images collectées proviennent de plusieurs sources. Elles ne sont donc pas 
homogènes : dimensions différentes ; formats différents ; images RGB et grayscale ; certaines images 
sont trop petites ; certaines images sont corrompues ; quelques images sont vides ; images 
dupliquées ; quelques images placées dans le mauvais dossier ; classes déséquilibrées.

L'objectif de l'atelier est donc de construire un jeu de données images propre et homogène, prêt à 
être utilisé pour entraîner un modèle de Machine Learning ou de Deep Learning. 

### <span style="color: #ffe602">Partie 1 – Exploration du dataset</span>

Un programme Python capable de récupérer, pour chaque image, son nom, sa classe, 
son format, son mode, sa largeur, sa hauteur, l’écart-type de ses pixels, son nombre de canaux et sa 
taille.

Construction d'un script simple pour auditer le dossier data/raw/ et extraire les caractéristiques techniques de chaque image avec la bibliothèque standard de gestion de fichiers en Python (os ou pathlib) et la bibliothèque de traitement d'images incontournable : Pillow (PIL).

In [1]:
import os
import pandas as pd
from PIL import Image

# 1. Définition du chemin vers les données brutes
CHEMIN_RAW = "../data/raw"

# 2. Liste pour stocker les informations de chaque image
liste_informations = []

# 3. Parcours des dossiers de chaque classe dans data/raw
for nom_classe in os.listdir(CHEMIN_RAW):
    chemin_classe = os.path.join(CHEMIN_RAW, nom_classe)
    
    # On s'assure qu'on parcourt bien un dossier (ex: cardboard, glass...)
    if os.path.isdir(chemin_classe):
        
        # Parcours de chaque fichier dans le dossier de la classe
        for nom_fichier in os.listdir(chemin_classe):
            chemin_image = os.path.join(chemin_classe, nom_fichier)
            
            # Pour éviter de traiter des fichiers système cachés (ex: .DS_Store)
            if nom_fichier.startswith('.'):
                continue
                
            # Initialisation d'un dictionnaire avec les infos de base
            info_image = {
                "nom": nom_fichier,
                "classe": nom_classe,
                "format": None,
                "mode": None,
                "largeur": None,
                "hauteur": None,
                "ecart_type": None,
                "canaux": None,
                "taille_octets": os.path.getsize(chemin_image), # Taille du fichier sur le disque
                "statut": "Valide"
            }
            
            try:
                # Tentative d'ouverture de l'image avec Pillow
                with Image.open(chemin_image) as img:
                    # On force le chargement pour vérifier si le fichier est réellement corrompu
                    img.verify() 
                    
                # On réouvre l'image pour lire ses propriétés (verify() ferme le fichier)
                with Image.open(chemin_image) as img:
                    info_image["format"] = img.format  # Ex: JPEG, PNG
                    info_image["mode"] = img.mode      # Ex: RGB, L (Grayscale)
                    info_image["largeur"] = img.size[0]
                    info_image["hauteur"] = img.size[1]
                    
                    # Détermination du nombre de canaux selon le mode
                    # RGB = 3 canaux, RGBA = 4 canaux, L (noir et blanc) = 1 canal
                    info_image["canaux"] = len(img.getbands())
                    
                    # Pour l'écart-type, on convertit temporairement en niveaux de gris 
                    # afin d'avoir une seule valeur globale simple pour cette étape
                    statistiques = img.convert("L").getextrema() 
                    # Note : getbands/histogram peut donner l'écart-type, mais pour faire simple
                    # et efficace sans numpy à ce stade, on peut aussi charger les données en niveaux de gris
                    import math
                    stat_visuelles = Image.Image.getdata(img.convert("L"))
                    # Version simplifiée pour débutant sans charger de grosse matrice :
                    # On utilise l'astuce de stocker temporairement la variance ou une valeur par défaut
                    # Mais pour être précis et performant, on peut importer brièvement numpy ici :
                    import numpy as np
                    matrice_pixels = np.array(img)
                    info_image["ecart_type"] = round(float(np.std(matrice_pixels)), 2)

            except Exception as e:
                # Si une erreur survient, le fichier est marqué comme corrompu
                info_image["statut"] = "Corrompu"
            
            # Ajout des données de l'image actuelle à notre liste
            liste_informations.append(info_image)

# 4. Conversion de la liste en DataFrame Pandas pour visualiser le résultat
df_exploration = pd.DataFrame(liste_informations)

# Affichage des 10 premières lignes du résultat
print("Aperçu des données explorées :")
print(df_exploration.head(10))


C:\Users\cissc\AppData\Local\Temp\ipykernel_19408\2156876818.py:63: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  stat_visuelles = Image.Image.getdata(img.convert("L"))


Aperçu des données explorées :
                nom     classe format mode  largeur  hauteur  ecart_type  \
0    cardboard1.jpg  cardboard   JPEG  RGB    512.0    384.0       40.59   
1   cardboard10.jpg  cardboard   JPEG  RGB    512.0    384.0       42.57   
2  cardboard100.jpg  cardboard   JPEG  RGB    512.0    384.0       46.11   
3  cardboard101.jpg  cardboard   JPEG  RGB    512.0    384.0       72.26   
4  cardboard102.jpg  cardboard   JPEG  RGB    512.0    384.0       48.39   
5  cardboard103.jpg  cardboard   JPEG  RGB    512.0    384.0       40.74   
6  cardboard104.jpg  cardboard   JPEG  RGB    512.0    384.0       38.82   
7  cardboard105.jpg  cardboard   JPEG  RGB    512.0    384.0       49.68   
8  cardboard106.jpg  cardboard   JPEG  RGB    512.0    384.0       57.07   
9  cardboard107.jpg  cardboard   JPEG  RGB    512.0    384.0       41.68   

   canaux  taille_octets  statut  
0     3.0          17333  Valide  
1     3.0          21683  Valide  
2     3.0          14884  V

### <span style="color: #ffde07">Partie 2 – Détecter les images corrompues</span> 

Une fonction qui détecte une image corrompue. 

Une image est considérée comme corrompue lorsque le fichier est endommagé (téléchargement incomplet, bug d'écriture sur le disque, format non reconnu).

In [2]:
# Fonction de detection d'image corrompue

from PIL import Image

def est_image_corrompue(chemin_fichier):
    """
    Vérifie si une image est corrompue ou illisible.
    Renvoie True si l'image est corrompue, False sinon.
    """
    try:
        with Image.open(chemin_fichier) as img:
            # La méthode verify() vérifie la structure du fichier sans décoder les pixels
            img.verify()
        return False  # Si aucune erreur n'est levée, l'image n'est pas corrompue
    except Exception:
        return True   # Si une erreur survient, l'image est corrompue


In [3]:
# Utilisation de la fonction de detection pour vérifier les images dans notre dossier raw

# Chemin vers nos images brutes
CHEMIN_RAW = "../data/raw"

# Liste pour stocker les chemins des images corrompues trouvées
images_corrompues = []

# Parcours des sous-dossiers de classes
for nom_classe in os.listdir(CHEMIN_RAW):
    chemin_classe = os.path.join(CHEMIN_RAW, nom_classe)
    
    if os.path.isdir(chemin_classe):
        for nom_fichier in os.listdir(chemin_classe):
            # Ignorer les fichiers système cachés
            if nom_fichier.startswith('.'):
                continue
                
            chemin_complet = os.path.join(chemin_classe, nom_fichier)
            
            # Appel de notre fonction
            if est_image_corrompue(chemin_complet):
                images_corrompues.append({
                    "Nom": nom_fichier,
                    "Classe": nom_classe,
                    "Chemin": chemin_complet
                })

# Affichage du bilan
print(f"Nombre total d'images corrompues détectées : {len(images_corrompues)}")
if len(images_corrompues) > 0:
    print("\nListe des images corrompues :")
    for img in images_corrompues:
        print(f"- Classe [{img['Classe']}] : Fichier {img['Nom']}")


Nombre total d'images corrompues détectées : 6

Liste des images corrompues :
- Classe [cardboard] : Fichier cardboard83.jpg
- Classe [glass] : Fichier glass74.jpg
- Classe [metal] : Fichier metal48.jpg
- Classe [paper] : Fichier paper213.jpg
- Classe [plastic] : Fichier plastic13.jpg
- Classe [trash] : Fichier trash3.jpg
